In [126]:
import pandas as pd
import duckdb
from datetime import datetime
from scipy import stats

In [134]:
path="C:\\Users\\TWB\\Desktop\\openClassrooms\\Projet_10\\data\\Flow01\\"
#path="C:\\Users\\ardit\\Desktop\\Projet_10\\data\\Flow01\\"

hashFiles = pd.read_excel(path + "hashFiles.xlsx", sheet_name="Sheet1")
erp = pd.read_excel(path + "Fichier_erp.xlsx", sheet_name="Sheet1")
web = pd.read_excel(path + "Fichier_web.xlsx", sheet_name="Sheet1")
liaison = pd.read_excel(path + "Fichier_liaison.xlsx", sheet_name="Sheet1")
CA = pd.read_excel(path + "chiffreAffaireTotal.xlsx", sheet_name="Sheet1")
#final = pd.read_excel(path + "FichierFinal.xlsx", sheet_name="Sheet1")
vinsMillesimes = pd.read_csv(path + "vinsMillesimes.csv")
final = pd.read_excel("C:\\data\\output.xlsx", sheet_name="Sheet1")


c:\Users\TWB\Desktop\openClassrooms\Projet_9\Projet_10\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
c:\Users\TWB\Desktop\openClassrooms\Projet_9\Projet_10\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
c:\Users\TWB\Desktop\openClassrooms\Projet_9\Projet_10\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


In [112]:
conn = duckdb.connect()
conn.sql("ATTACH 'openclassrooms.db'")

# Comparaison des deux derniers hash avec la date
print(hashFiles.loc[hashFiles.shape[0] - 1, "hashFileErp"])
print(hashFiles.loc[hashFiles.shape[0] - 2, "hashFileErp"])

print(hashFiles.loc[hashFiles.shape[0] - 1, "hashFileWeb"])
print(hashFiles.loc[hashFiles.shape[0] - 2, "hashFileWeb"])

print(hashFiles.loc[hashFiles.shape[0] - 1, "hashFileLiaison"])
print(hashFiles.loc[hashFiles.shape[0] - 2, "hashFileLiaison"])

# Comparaison de la date
cdt = datetime.now()
todayDate = '%s/%s/%s' % (cdt.day, cdt.month, cdt.year)
print(todayDate)
print(hashFiles.loc[hashFiles.shape[0] - 1, "date"])



conn.sql("DETACH openclassrooms")# instanciation de la date
conn.close()

5cb660fb25650eb3ce779a4d35aa858e
5cb660fb25650eb3ce779a4d35aa858e
10a15ca8c8659504c4380bc188b4bf20
10a15ca8c8659504c4380bc188b4bf20
5cb660fb25650eb3ce779a4d35aa858e
5cb660fb25650eb3ce779a4d35aa858e
14/3/2026
14/3/2026


## Script de récupération d'infrmations des fichiers

In [ ]:
# Afin de tester si l'export s'est bien déroulé :
# - on recupére le hash des anciens et on les compare aux nouveaux fichiers et on vérifie qu'ils sont différents
# - on vérifie la colonne dateCreation des fichiers et on verifie qu'il s'agit bien de ce jour
# - On récupère le nombre de lignes 
# - On exporte le résultat dans un fichiers Excel, qui est mis à jour à chaque exécution du flow

# table à créer
# idExport
# hashErp
# hashWeb
# hashLiaison
# nbDoublons
# nbValManquantes
# jointureErp
# jointureWeb
# chiffreAffaireErp
# chiffreAffaireRetenu
# Z-scoreErp
# Z-scoreRetenu

conn = duckdb.connect()
conn.sql("ATTACH 'openclassrooms.db'")


conn.sql("DROP TABLE IF EXISTS openclassrooms.resultExtract")
conn.sql("CREATE TABLE openclassrooms.resultExtract (idExport VARCHAR, hashErp VARCHAR, hashWeb VARCHAR, hashLiaison VARCHAR, nbDoublons INTEGER, colValManquantes VARCHAR, jointureIncoherentes INTEGER, chiffreAffaireApresJointure varchar, ZscoreErpMillesimes INTEGER, ZscoreApresJointureMillesimes INTEGER);")

idExport = final[['idExport']].drop_duplicates().iloc[0]["idExport"]
hashErp = hashFiles.loc[hashFiles.shape[0] - 1, "hashFileErp"]
hashWeb = hashFiles.loc[hashFiles.shape[0] - 1, "hashFileWeb"]
hashLiaison = hashFiles.loc[hashFiles.shape[0] - 1, "hashFileLiaison"]

# Insertion des données dans DUckDB
conn.sql("INSERT INTO openclassrooms.resultExtract VALUES ('" + idExport + "','" + hashErp + "','" + hashWeb + "','" + hashLiaison + "',NULL,NULL,NULL,NULL,NULL,NULL);")
 
conn.sql("DETACH openclassrooms")
conn.close()

┌────────────────────────────┬──────────────────────────────────┬──────────────────────────────────┬──────────────────────────────────┬────────────┬──────────────────┬──────────────────────┬─────────────────────────────┬─────────────────────┬───────────────────────────────┐
│          idExport          │             hashErp              │             hashWeb              │           hashLiaison            │ nbDoublons │ colValManquantes │ jointureIncoherentes │ chiffreAffaireApresJointure │ ZscoreErpMillesimes │ ZscoreApresJointureMillesimes │
│          varchar           │             varchar              │             varchar              │             varchar              │   int32    │     varchar      │        int32         │           varchar           │        int32        │             int32             │
├────────────────────────────┼──────────────────────────────────┼──────────────────────────────────┼──────────────────────────────────┼────────────┼──────────────────┼────────

## Script de test d'absence de doublons

In [ ]:
# Récupérer le fichier final et tester l'absence de doublons des identifiants

conn = duckdb.connect()
conn.sql("ATTACH 'openclassrooms.db'")

idExport = final[['idExport']].drop_duplicates().iloc[0]["idExport"]

# Vérification qu'il n'y a pas de doublons dans le fichier final (avec product_id et id_web)
nbDoublons = final["product_id"].shape[0] - final["product_id"].drop_duplicates().shape[0]
nbDoublons = nbDoublons + (final["id_web"].shape[0] - final["id_web"].drop_duplicates().shape[0])

# Insertion des données dans DUckDB
conn.sql("UPDATE openclassrooms.resultExtract SET nbDoublons = " + str(nbDoublons) + " where idExport = '"+ idExport + "' ;")

conn.sql("DETACH openclassrooms")
conn.close()

┌────────────────────────────┬──────────────────────────────────┬──────────────────────────────────┬──────────────────────────────────┬────────────┬──────────────────┬──────────────────────┬─────────────────────────────┬─────────────────────┬───────────────────────────────┐
│          idExport          │             hashErp              │             hashWeb              │           hashLiaison            │ nbDoublons │ colValManquantes │ jointureIncoherentes │ chiffreAffaireApresJointure │ ZscoreErpMillesimes │ ZscoreApresJointureMillesimes │
│          varchar           │             varchar              │             varchar              │             varchar              │   int32    │     varchar      │        int32         │           varchar           │        int32        │             int32             │
├────────────────────────────┼──────────────────────────────────┼──────────────────────────────────┼──────────────────────────────────┼────────────┼──────────────────┼────────

## Script de test d'absence de valeurs manquantes

In [ ]:
# Faire un résume des valeurs manquantes par colonnes

conn = duckdb.connect()
conn.sql("ATTACH 'openclassrooms.db'")

idExport = final[['idExport']].drop_duplicates().iloc[0]["idExport"]

# Récupértion des colonnes qui ont une valeur nulle
columnWithNullValue = ""
for series_name, series in final.items():
    if(series.loc[series.isna()].shape[0] > 0) :
        columnWithNullValue = columnWithNullValue + " - " +  series_name
columnWithNullValue = columnWithNullValue + " - "

# Insertion des données dans DUckDB
conn.sql("UPDATE openclassrooms.resultExtract SET colValManquantes = '" + columnWithNullValue + "' where idExport = '"+ idExport + "' ;")

conn.sql("DETACH openclassrooms")
conn.close()

┌────────────────────────────┬──────────────────────────────────┬──────────────────────────────────┬──────────────────────────────────┬────────────┬────────────────────────────────────────────────────────────────────────┬──────────────────────┬─────────────────────────────┬─────────────────────┬───────────────────────────────┐
│          idExport          │             hashErp              │             hashWeb              │           hashLiaison            │ nbDoublons │                            colValManquantes                            │ jointureIncoherentes │ chiffreAffaireApresJointure │ ZscoreErpMillesimes │ ZscoreApresJointureMillesimes │
│          varchar           │             varchar              │             varchar              │             varchar              │   int32    │                                varchar                                 │        int32         │           varchar           │        int32        │             int32             │
├────────────

## Script de test de cohérence des jointures

In [ ]:
# Vérifier la cohérence des jointures avec le fichier "Fichier_liaison" et faire un résumé des lignes non jointes

conn = duckdb.connect()
conn.sql("ATTACH 'openclassrooms.db'")

idExport = final[['idExport']].drop_duplicates().iloc[0]["idExport"]

# Vérification cohérence jointure
idWebErp = pd.DataFrame()
idWebErp["idJointure"] = final["product_id"].astype(str) + "-" + final["id_web"].astype(str)

idLiaison = pd.DataFrame()
idLiaison["idJointure"]  = liaison["product_id"].astype(str) + "-" + liaison["id_web"].astype(str)

# print(idWebErp["idJointure"].isin(idLiaison["idJointure"]).shape[0])
# print(final.shape[0])
jointuresIncoherents = final.shape[0] - idWebErp["idJointure"].isin(idLiaison["idJointure"]).shape[0]
print(jointuresIncoherents)

# Insertion des données dans DUckDB
conn.sql("UPDATE openclassrooms.resultExtract SET jointureIncoherentes = '" + str(jointuresIncoherents) + "' where idExport = '"+ idExport + "' ;")

conn.sql("DETACH openclassrooms")
conn.close()

0
┌────────────────────────────┬──────────────────────────────────┬──────────────────────────────────┬──────────────────────────────────┬────────────┬────────────────────────────────────────────────────────────────────────┬──────────────────────┬─────────────────────────────┬─────────────────────┬───────────────────────────────┐
│          idExport          │             hashErp              │             hashWeb              │           hashLiaison            │ nbDoublons │                            colValManquantes                            │ jointureIncoherentes │ chiffreAffaireApresJointure │ ZscoreErpMillesimes │ ZscoreApresJointureMillesimes │
│          varchar           │             varchar              │             varchar              │             varchar              │   int32    │                                varchar                                 │        int32         │           varchar           │        int32        │             int32             │
├──────────

## Script de test de cohérence du chiffre d'affaire

In [ ]:
# Calculer le chiffre d'affaire après jointure avec Pandas
conn = duckdb.connect()
conn.sql("ATTACH 'openclassrooms.db'")

idExport = final[['idExport']].drop_duplicates().iloc[0]["idExport"]

# Comparaison du chiffre d'affaire obtenu lors de l'export
result = ""
if(CA["sum(chiffreAffaire)"][0].astype("int") == int(sum(final["total_sales"] * final["price"]))) :
    result = str(int(sum(final["total_sales"] * final["price"])))
else :
    result = "error"

# Insertion des données dans DUckDB
conn.sql("UPDATE openclassrooms.resultExtract SET chiffreAffaireApresJointure = " + result + " where idExport = '" + idExport + "' ;")

conn.sql("DETACH openclassrooms")
conn.close()

70568
┌────────────────────────────┬──────────────────────────────────┬──────────────────────────────────┬──────────────────────────────────┬────────────┬────────────────────────────────────────────────────────────────────────┬──────────────────────┬─────────────────────────────┬─────────────────────┬───────────────────────────────┐
│          idExport          │             hashErp              │             hashWeb              │           hashLiaison            │ nbDoublons │                            colValManquantes                            │ jointureIncoherentes │ chiffreAffaireApresJointure │ ZscoreErpMillesimes │ ZscoreApresJointureMillesimes │
│          varchar           │             varchar              │             varchar              │             varchar              │   int32    │                                varchar                                 │        int32         │           varchar           │        int32        │             int32             │
├──────

## Script de test du Z-score

In [ ]:
# Faire le test du Z-score sur le "Fichier_erp" et le comparer avec le resultat final et dire pourquoi il y'a une différence

conn = duckdb.connect()
conn.sql("ATTACH 'openclassrooms.db'")

idExport = final[['idExport']].drop_duplicates().iloc[0]["idExport"]

# Calcul du Z-score avec le fichier Erp
serie_Z_erp = pd.Series(stats.zscore(erp["price"]))
resultErp = str(serie_Z_erp.loc[serie_Z_erp > 2].count())

# Comparaison du Z-score avec le Z-score obtenu par l'export
resultFinal = ""
serie_Z_final = pd.Series(stats.zscore(final["price"]))
if(vinsMillesimes.shape[0] == serie_Z_final.loc[serie_Z_final > 2].count()) :
    resultFinal = str(vinsMillesimes.shape[0])
else :
    resultFinal = "error"
    
# Insertion des données dans DUckDB
conn.sql("UPDATE openclassrooms.resultExtract SET ZscoreErpMillesimes = " + resultErp + " where idExport = '"+ idExport + "' ;")
conn.sql("UPDATE openclassrooms.resultExtract SET ZscoreApresJointureMillesimes = " + resultFinal + " where idExport = '"+ idExport + "' ;")

conn.sql("DETACH openclassrooms")
conn.close()

┌────────────────────────────┬──────────────────────────────────┬──────────────────────────────────┬──────────────────────────────────┬────────────┬────────────────────────────────────────────────────────────────────────┬──────────────────────┬─────────────────────────────┬─────────────────────┬───────────────────────────────┐
│          idExport          │             hashErp              │             hashWeb              │           hashLiaison            │ nbDoublons │                            colValManquantes                            │ jointureIncoherentes │ chiffreAffaireApresJointure │ ZscoreErpMillesimes │ ZscoreApresJointureMillesimes │
│          varchar           │             varchar              │             varchar              │             varchar              │   int32    │                                varchar                                 │        int32         │           varchar           │        int32        │             int32             │
├────────────

## Export xlsx

In [ ]:
conn = duckdb.connect()
conn.sql("ATTACH 'openclassrooms.db'")

# Existing Excel file
testsExport = 'C:\\Users\\TWB\\Desktop\\openClassrooms\\Projet_10\\data\\Flow01\\testsExport.xlsx'

idExport = final[['idExport']].drop_duplicates().iloc[0]["idExport"]

# # Today date
# cdt = datetime.now()
# todayDate = '%s/%s/%s' % (cdt.day, cdt.month, cdt.year)
# #The comment `# select de la date` is likely indicating a step where the code is selecting the current date. However, the actual code snippet related to selecting the date is missing in the provided code.

# New data to append
df_new = conn.sql("SELECT * FROM openclassrooms.resultExtract where idExport = '" + idExport + "' ;").df()


# Read existing data
df_existing = pd.read_excel(testsExport)
# Append new data
df_combined = pd.concat([df_existing, df_new])
# Save the combined data to Excel
df_combined.to_excel(testsExport, index=False)


conn.sql("DETACH openclassrooms")
conn.close()

                     idExport                           hashErp  \
0  2026-03-14 14:15:36.632121  5cb660fb25650eb3ce779a4d35aa858e   

                            hashWeb                       hashLiaison  \
0  10a15ca8c8659504c4380bc188b4bf20  5cb660fb25650eb3ce779a4d35aa858e   

   nbDoublons                                   colValManquantes  \
0           0   - tax_class - post_content - post_password - ...   

   jointureIncoherentes chiffreAffaireApresJointure  ZscoreErpMillesimes  \
0                     0                       70568                   34   

   ZscoreApresJointureMillesimes  
0                             30  


## Envoie des informations par email

In [ ]:
# Envoyer ces informations par email à chaque exécution du flux